# TraceGuard V4 — Planner Evaluation Runner

Tests the Agent Planner against 15–20 real engineering queries, drawn from
actual artifacts in the dataset. Each query is chosen to exercise a
specific part of the routing pipeline:

- **Known-entity, free-phrased "understand/explain" queries** — deliberately
  written to avoid every existing lexical rule (over 6 words, no
  baseline/trace/similarity/impact keyword), so they either get caught by
  semantic fallback or reach the Planner.
- **Free-text engineering change descriptions** — no artifact ID at all,
  testing whether the Planner (or semantic fallback) correctly proposes
  `full_impact_analysis` rather than a manual multi-step chain.
- **Genuinely irrelevant queries** — sanity checks that `clarification`
  still fires correctly even with the Planner available as a fallback.
- **Known-workflow controls** — queries that *should* hit the fast lexical
  path, included so you can see the contrast between "fast path" and
  "planner path" side by side.



In [1]:
import sys
from pathlib import Path

# Notebook is expected under <repo>/notebooks and module under <repo>/src.
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.traceguard_v2 import TraceGuard
from src.orchestrator import Orchestrator

traceguard = TraceGuard(data_path=repo_root / 'data')
orch = Orchestrator(traceguard)
print('TraceGuard + Orchestrator ready.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

TraceGuard + Orchestrator ready.


## Test Queries

Each entry has a `query` and a `hypothesis` — what pattern this query is
meant to test. The hypothesis is a prediction, not a guarantee: whether a
"planner-target" query actually reaches the Planner depends on real
semantic-fallback scores, which can only be checked by running it (that's
the whole point of this notebook).

In [2]:
TEST_QUERIES = [
    # --- Known-entity, free-phrased "understand/explain" queries ---
    # (word count > 6, zero keyword hits -- verified against the real
    # rules before writing this notebook, not guessed)
    {"query": "Can you walk me through everything about CR-00319 in detail",
     "hypothesis": "Planner: lookup -> trace (explain a known CR)"},
    {"query": "I need a full picture of PR-00146 before the review meeting",
     "hypothesis": "Planner: lookup -> trace (explain a known PR)"},
    {"query": "Please provide comprehensive background on REQ-00269 for the audit",
     "hypothesis": "Planner: lookup -> trace (explain a known Requirement)"},
    {"query": "Give me an in-depth explanation of what TC-00621 actually covers",
     "hypothesis": "Planner: lookup -> trace (explain a known Test Case)"},
    {"query": "Walk me through the purpose and scope of TASK-00187 please",
     "hypothesis": "Planner: lookup -> trace (explain a known Task)"},

    # --- Known-entity, baseline/release-flavored WITHOUT the literal keyword ---
    {"query": "Does shipping CR-00741 require checking anything about the current build",
     "hypothesis": "Planner: does it infer baseline_evidence without the word 'baseline'?"},
    {"query": "I want to know if PR-00467 has any downstream shipment implications",
     "hypothesis": "Planner: same test, release/shipment-flavored phrasing"},

    # --- Known-entity, traceability-flavored WITHOUT the literal keyword ---
    {"query": "What upstream and downstream connections exist for SPEC-00711",
     "hypothesis": "Planner: does it infer trace without the word 'traceability'?"},
    {"query": "Show me everything that feeds into or comes out of REQ-00066",
     "hypothesis": "Planner: same test, different phrasing"},

    # --- Free-text engineering change descriptions, no artifact ID ---
    {"query": "We need to enhance charge port locking plausibility to support improved connector durability",
     "hypothesis": "Planner or semantic fallback: should propose full_impact_analysis, not a manual chain"},
    {"query": "Please look into incorrect fallback behavior for battery thermal protection under transient load",
     "hypothesis": "Planner or semantic fallback: full_impact_analysis"},
    {"query": "Investigate coolant flow improvements within the current thermal management architecture",
     "hypothesis": "Planner or semantic fallback: full_impact_analysis"},
    {"query": "Assess gateway routing adjustments needed in the vehicle network topology",
     "hypothesis": "Planner or semantic fallback: full_impact_analysis"},

    # --- Genuinely irrelevant -- should still land in clarification ---
    {"query": "hey",
     "hypothesis": "Clarification (irrelevant, too short/vague for any plan)"},
    {"query": "can I get a coffee recommendation nearby",
     "hypothesis": "Clarification (irrelevant to the engineering domain)"},
    {"query": "what's on the agenda for tomorrow's standup",
     "hypothesis": "Clarification (irrelevant to the engineering domain)"},

    # --- Known-workflow controls: should hit the FAST lexical path ---
    {"query": "Baseline impact of CR-00319",
     "hypothesis": "Known workflow: baseline_check (lexical, 'baseline' keyword)"},
    {"query": "Change requirements for the diagnostic monitoring system",
     "hypothesis": "Known workflow: full_impact_analysis (lexical, 'change' keyword)"},
    {"query": "Show traceability for SPEC-00711",
     "hypothesis": "Known workflow: traceability_trace (lexical, 'traceability' keyword)"},
    {"query": "What is TC-00621",
     "hypothesis": "Known workflow: direct_lookup (lexical, entity + <=6 words shortcut)"},
]

print(f"{len(TEST_QUERIES)} test queries loaded.")

20 test queries loaded.


## Run All Queries

In [3]:
results = []

for item in TEST_QUERIES:
    query = item["query"]
    r = orch.run(query)
    results.append({
        "query": query,
        "hypothesis": item["hypothesis"],
        "workflow": r.workflow,
        "confidence": round(r.confidence, 3),
        "steps_run": r.steps_run,
        "success": r.success,
        "plan_reasoning": r.plan_reasoning,
        "execution_time_s": round(r.execution_time_seconds, 3),
    })
    print(f"[{'OK ' if r.success else 'FAIL'}] {query[:65]:65} -> {r.workflow}")

print(f"\nDone. {sum(1 for x in results if x['success'])}/{len(results)} succeeded.")

[OK ] Can you walk me through everything about CR-00319 in detail       -> direct_lookup
[OK ] I need a full picture of PR-00146 before the review meeting       -> direct_lookup
[OK ] Please provide comprehensive background on REQ-00269 for the audi -> direct_lookup
[OK ] Give me an in-depth explanation of what TC-00621 actually covers  -> direct_lookup
[OK ] Walk me through the purpose and scope of TASK-00187 please        -> direct_lookup
[OK ] Does shipping CR-00741 require checking anything about the curren -> baseline_check
[OK ] I want to know if PR-00467 has any downstream shipment implicatio -> traceability_trace
[OK ] What upstream and downstream connections exist for SPEC-00711     -> traceability_trace
[OK ] Show me everything that feeds into or comes out of REQ-00066      -> direct_lookup
[OK ] We need to enhance charge port locking plausibility to support im -> full_impact_analysis
[OK ] Please look into incorrect fallback behavior for battery thermal  -> similarity_check


## Results Table

In [4]:
import pandas as pd

results_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', 60)
results_df[["query", "hypothesis", "workflow", "confidence", "steps_run", "success"]]

,query,hypothesis,workflow,confidence,steps_run,success
0,Can you walk me through everything about CR-00319 in detail,Planner: lookup -> trace (explain a known CR),direct_lookup,0.377,[lookup],True
1,I need a full picture of PR-00146 before the review meeting,Planner: lookup -> trace (explain a known PR),direct_lookup,0.380,[lookup],True
2,Please provide comprehensive background on REQ-00269 for...,Planner: lookup -> trace (explain a known Requirement),direct_lookup,0.341,[lookup],True
3,Give me an in-depth explanation of what TC-00621 actuall...,Planner: lookup -> trace (explain a known Test Case),direct_lookup,0.520,[lookup],True
4,Walk me through the purpose and scope of TASK-00187 please,Planner: lookup -> trace (explain a known Task),direct_lookup,0.359,[lookup],True
5,Does shipping CR-00741 require checking anything about t...,Planner: does it infer baseline_evidence without the wor...,baseline_check,0.304,"[lookup, trace, baseline_evidence]",True
6,I want to know if PR-00467 has any downstream shipment i...,"Planner: same test, release/shipment-flavored phrasing",traceability_trace,0.401,"[lookup, trace]",True
7,What upstream and downstream connections exist for SPEC-...,Planner: does it infer trace without the word 'traceabil...,traceability_trace,0.430,"[lookup, trace]",True
8,Show me everything that feeds into or comes out of REQ-0...,"Planner: same test, different phrasing",direct_lookup,0.281,[lookup],True
9,We need to enhance charge port locking plausibility to s...,Planner or semantic fallback: should propose full_impact...,full_impact_analysis,0.446,"[full_impact_analysis, baseline_evidence, evidence_fusion]",True


## Demo: Different Queries, Different Plans

A curated, presentation-friendly view of a few illustrative queries —
useful for screenshots or a live walkthrough. Shows the query, which path
it took, the actual steps executed, and (for Planner-composed plans) the
LLM's own stated reasoning.

In [5]:
DEMO_INDICES = [0, 5, 9, 13, 16, 19]  # a representative mix across every category above

for i in DEMO_INDICES:
    r = results[i]
    print("=" * 78)
    print(f"QUERY:      {r['query']}")
    print(f"WORKFLOW:   {r['workflow']}")
    print(f"STEPS:      {' -> '.join(r['steps_run']) if r['steps_run'] else '(none)'}")
    if r["plan_reasoning"]:
        print(f"REASONING:  {r['plan_reasoning']}")
    print(f"SUCCESS:    {r['success']}")
    print()

QUERY:      Can you walk me through everything about CR-00319 in detail
WORKFLOW:   direct_lookup
STEPS:      lookup
SUCCESS:    True

QUERY:      Does shipping CR-00741 require checking anything about the current build
WORKFLOW:   baseline_check
STEPS:      lookup -> trace -> baseline_evidence
SUCCESS:    True

QUERY:      We need to enhance charge port locking plausibility to support improved connector durability
WORKFLOW:   full_impact_analysis
STEPS:      full_impact_analysis -> baseline_evidence -> evidence_fusion
SUCCESS:    True

QUERY:      hey
WORKFLOW:   planner:hey
STEPS:      retrieve -> full_impact_analysis
REASONING:  The user's query is free-text and does not specify an artifact ID, necessitating a full impact analysis to effectively address the request.
SUCCESS:    True

QUERY:      Baseline impact of CR-00319
WORKFLOW:   baseline_check
STEPS:      lookup -> trace -> baseline_evidence
SUCCESS:    True

QUERY:      What is TC-00621
WORKFLOW:   direct_lookup
STEPS:      l

## Evaluation Summary

In [6]:
known_workflows = {"direct_lookup", "similarity_check", "traceability_trace", "baseline_check", "full_impact_analysis"}

fast_path = sum(1 for x in results if x["workflow"] in known_workflows)
planner_path = sum(1 for x in results if x["workflow"].startswith("planner:"))
clarification_path = sum(1 for x in results if x["workflow"] == "clarification")

print(f"Fast lexical/semantic path (known workflow): {fast_path}")
print(f"Planner-composed path:                       {planner_path}")
print(f"Clarification (no confident route/plan):     {clarification_path}")
print(f"Total:                                        {len(results)}")

failures = [x for x in results if not x["success"]]
if failures:
    print(f"\n{len(failures)} query(ies) did not succeed -- review these first:")
    for f in failures:
        print(f"  - {f['query']!r} -> {f['workflow']}")
else:
    print("\nAll queries executed successfully (this does not mean every plan was" 
          " the BEST plan -- check the table above against each hypothesis by eye).")

Fast lexical/semantic path (known workflow): 15
Planner-composed path:                       5
Clarification (no confident route/plan):     0
Total:                                        20

All queries executed successfully (this does not mean every plan was the BEST plan -- check the table above against each hypothesis by eye).


## Manual Review Notes

Automated success/failure only tells you a plan *ran* without error — not
whether it was the *right* plan. For each row in the results table above,
worth eyeballing against its `hypothesis`:

- **Did known-entity "explain/understand" queries actually get `lookup -> trace`**,
  or something longer/shorter than expected?
- **Did the baseline/traceability-flavored queries (without the literal keyword)**
  get routed sensibly, or did they need the keyword to work at all?
- **Did free-text engineering descriptions correctly prefer `full_impact_analysis`**
  over a manually recreated `retrieve -> trace -> assess_impact -> ...` chain?
- **Did every genuinely irrelevant query land in `clarification`**, with no
  Planner-composed plan sneaking through?
